### Cell 1 — Imports and configuration


In [ ]:
# Install missing dependencies into active kernel
%pip install statsmodels scikit-learn numpy pandas matplotlib

# ============================================================
# NOTEBOOK 4 — ABSOLUTE THRESHOLD SENSITIVITY
# ============================================================
#
# Scientific question:
#
#   Is the empirical tau_BP scaling robust to reasonable changes
#   in the absolute gradient-variance threshold?
#
# Primary threshold:
#   1e-2
#
# Sensitivity thresholds:
#   1e-1, 5e-2, 2e-2, 1e-2, 5e-3, 1e-3, 1e-4
#
# IMPORTANT:
#   - No quantum simulation.
#   - No hardcoded fitted A or c.
#   - tau_BP is re-derived directly from raw variance curves.
#   - n=14 and n=16 are excluded from fitting.
#   - Censoring is explicitly reported.
#   - The 1e-2 result is treated as the baseline reference only.
#
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

INPUT_FILE = "variance_curves.csv"
# Ensure input file is found whether running from Downloads or scratch folder
if not os.path.exists(INPUT_FILE):
    for _cand in [r"C:\Users\Abhishek\Downloads\prevntnbkoutputs\variance_curves.csv", r"C:\Users\Abhishek\Downloads\variance_curves.csv"]:
        if os.path.exists(_cand):
            INPUT_FILE = _cand
            break

BASELINE_THRESHOLD = 1e-2

THRESHOLDS = [
    1e-1,
    5e-2,
    2e-2,
    1e-2,
    5e-3,
    1e-3,
    1e-4,
]

TRAIN_N = [8, 10, 12]
HELD_OUT_N = [14, 16]

OUTPUT_DIR = "notebook4_threshold_sensitivity"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 80)
print("NOTEBOOK 4 — ABSOLUTE THRESHOLD SENSITIVITY")
print("=" * 80)
print(f"Input:               {INPUT_FILE}")
print(f"Baseline threshold:  {BASELINE_THRESHOLD}")
print(f"Sensitivity levels:  {THRESHOLDS}")
print(f"Training n:           {TRAIN_N}")
print(f"Held-out n:           {HELD_OUT_N}")
print("Quantum simulation:  NONE")
print("=" * 80)


### Cell 2 — Load and validate raw variance curves


In [ ]:
# ============================================================
# CELL 2 — LOAD RAW VARIANCE CURVES
# ============================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"CRITICAL: {INPUT_FILE} was not found."
    )

raw_df = pd.read_csv(INPUT_FILE)

required_columns = [
    "n",
    "k",
    "depth",
    "var_pooled",
]

missing = [
    col for col in required_columns
    if col not in raw_df.columns
]

if missing:
    raise ValueError(
        f"CRITICAL: Missing required columns: {missing}"
    )

for col in required_columns:
    raw_df[col] = pd.to_numeric(
        raw_df[col],
        errors="coerce"
    )

if raw_df[required_columns].isna().any().any():
    raise ValueError(
        "CRITICAL: Missing/non-numeric values detected "
        "in required columns."
    )

print("=" * 80)
print("RAW DATA LOADED")
print("=" * 80)

print(f"Rows: {len(raw_df):,}")
print(
    f"n values: "
    f"{sorted(raw_df['n'].astype(int).unique())}"
)
print(
    f"k values: "
    f"{sorted(raw_df['k'].astype(int).unique())}"
)
print(
    f"Depth range: "
    f"{int(raw_df['depth'].min())}–"
    f"{int(raw_df['depth'].max())}"
)

if "architecture" in raw_df.columns:
    print(
        "Architecture(s):",
        raw_df["architecture"].dropna().unique().tolist()
    )


### Cell 3 — Verify raw curves are compatible with threshold analysis


In [ ]:
# ============================================================
# CELL 3 — RAW CURVE INTEGRITY CHECK
# ============================================================

print("\n" + "=" * 80)
print("RAW CURVE INTEGRITY CHECK")
print("=" * 80)

# One architecture per (n,k)
if "architecture" in raw_df.columns:

    arch_counts = (
        raw_df.groupby(["n", "k"])["architecture"]
        .nunique()
    )

    bad_arch = arch_counts[arch_counts > 1]

    if len(bad_arch) > 0:
        raise ValueError(
            "CRITICAL: Multiple architectures occur within "
            f"the same (n,k):\n{bad_arch}"
        )

    print("PASS: One architecture per (n,k).")

# Duplicate depth records
dup_counts = (
    raw_df
    .groupby(["n", "k", "depth"])
    .size()
)

duplicates = dup_counts[dup_counts > 1]

if len(duplicates) > 0:
    raise ValueError(
        "CRITICAL: Duplicate (n,k,depth) measurements detected."
    )

print("PASS: No duplicate (n,k,depth) records.")

# Variance validity
if (raw_df["var_pooled"] < 0).any():
    raise ValueError(
        "CRITICAL: Negative variance value detected."
    )

print("PASS: All pooled variances are non-negative.")

# Depth continuity
coverage = []

for (n, k), group in raw_df.groupby(["n", "k"]):

    depths = sorted(
        group["depth"].astype(int).unique()
    )

    expected = set(
        range(min(depths), max(depths) + 1)
    )

    missing_depths = sorted(
        expected - set(depths)
    )

    coverage.append({
        "n": int(n),
        "k": int(k),
        "min_depth": int(min(depths)),
        "max_depth": int(max(depths)),
        "n_depths": len(depths),
        "missing_depths": missing_depths,
    })

coverage_df = pd.DataFrame(coverage)

if (
    coverage_df["missing_depths"]
    .apply(len)
    .gt(0)
    .any()
):

    raise ValueError(
        "CRITICAL: Missing internal depth records detected."
    )

print("PASS: All curves have contiguous depth coverage.")


### Cell 4 — Function to derive \(\tau_{BP}\) for any threshold


In [ ]:
# ============================================================
# CELL 4 — TAU EXTRACTION FUNCTION
# ============================================================

MAX_DEPTH = int(
    raw_df["depth"].max()
)


def derive_tau_dataset(raw_variance_df, threshold):
    """
    Derive tau_BP for every (n,k) configuration.

    Definition:
        tau_BP = first depth L for which
                 var_pooled(L) < threshold

    If the threshold is never crossed:
        tau_BP = MAX_DEPTH + 1
        censored = True
    """

    records = []

    for (n, k), group in raw_variance_df.groupby(
        ["n", "k"]
    ):

        group = group.sort_values(
            "depth"
        )

        crossed = group[
            group["var_pooled"] < threshold
        ]

        if len(crossed) > 0:

            tau = int(
                crossed.iloc[0]["depth"]
            )

            censored = False

        else:

            tau = MAX_DEPTH + 1
            censored = True

        architecture = (
            group["architecture"].iloc[0]
            if "architecture" in group.columns
            else "UNKNOWN"
        )

        records.append({
            "n": int(n),
            "k": int(k),
            "nk": int(n * k),
            "architecture": architecture,
            "threshold": float(threshold),
            "tau_BP": int(tau),
            "censored": bool(censored),
        })

    return pd.DataFrame(records)


### Cell 5 — Derive \(\tau_{BP}\) for every threshold


In [ ]:
# ============================================================
# CELL 5 — DERIVE TAU FOR ALL THRESHOLDS
# ============================================================

threshold_tau_tables = []

print("=" * 80)
print("DERIVING TAU_BP ACROSS THRESHOLDS")
print("=" * 80)

for threshold in THRESHOLDS:

    print(
        f"\nProcessing threshold = {threshold:.1e}"
    )

    tau_table = derive_tau_dataset(
        raw_df,
        threshold
    )

    threshold_tau_tables.append(
        tau_table
    )

    n_censored = int(
        tau_table["censored"].sum()
    )

    print(
        f"Configurations: {len(tau_table)}"
    )

    print(
        f"Censored: {n_censored}"
    )

    print(
        f"Censoring rate: "
        f"{100 * n_censored / len(tau_table):.2f}%"
    )

threshold_tau_df = pd.concat(
    threshold_tau_tables,
    ignore_index=True
)

print("\nAll thresholds processed.")


### Cell 6 — Show threshold-dependent \(\tau\) table


In [ ]:
# ============================================================
# CELL 6 — TAU TABLE BY THRESHOLD
# ============================================================

print("=" * 80)
print("TAU_BP AS A FUNCTION OF ABSOLUTE THRESHOLD")
print("=" * 80)

tau_pivot = (
    threshold_tau_df
    .pivot_table(
        index=["n", "k"],
        columns="threshold",
        values="tau_BP"
    )
    .reset_index()
)

display(tau_pivot)


### Cell 7 — Fit the product model independently at each threshold


In [ ]:
# ============================================================
# CELL 7 — FIT PRODUCT MODEL AT EACH THRESHOLD
# ============================================================
#
# Model:
#
#     tau_BP = A * (n*k)^(-c)
#
# fitted in log space:
#
#     log(tau_BP) = log(A) - c*log(n*k)
#
# Only uncensored TRAINING configurations are used.
# ============================================================

def fit_product_model(data):
    """
    Fit the empirical product model in log space.
    """

    fit_data = data[
        ~data["censored"]
    ].copy()

    if len(fit_data) < 3:
        raise ValueError(
            "Too few uncensored observations for a stable fit."
        )

    fit_data["log_tau"] = np.log(
        fit_data["tau_BP"].astype(float)
    )

    fit_data["log_nk"] = np.log(
        fit_data["nk"].astype(float)
    )

    X = sm.add_constant(
        fit_data["log_nk"]
    )

    model = sm.OLS(
        fit_data["log_tau"],
        X
    ).fit()

    log_A = float(
        model.params["const"]
    )

    slope = float(
        model.params["log_nk"]
    )

    A = float(
        np.exp(log_A)
    )

    c = float(
        -slope
    )

    pred_tau = (
        A
        * fit_data["nk"].astype(float)
        .pow(-c)
    )

    residual = (
        fit_data["tau_BP"]
        - pred_tau
    )

    mae = mean_absolute_error(
        fit_data["tau_BP"],
        pred_tau
    )

    rmse = np.sqrt(
        mean_squared_error(
            fit_data["tau_BP"],
            pred_tau
        )
    )

    r2 = r2_score(
        fit_data["tau_BP"],
        pred_tau
    )

    return {
        "model": model,
        "fit_data": fit_data,
        "A": A,
        "c": c,
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2_tau": float(r2),
    }


threshold_fit_rows = []
threshold_fit_objects = {}

for threshold in THRESHOLDS:

    threshold_df = threshold_tau_df[
        threshold_tau_df["threshold"] == threshold
    ].copy()

    train_threshold_df = threshold_df[
        threshold_df["n"].isin(TRAIN_N)
    ].copy()

    n_total = len(train_threshold_df)

    n_censored = int(
        train_threshold_df["censored"].sum()
    )

    n_used = n_total - n_censored

    result = {
        "threshold": threshold,
        "n_training_total": n_total,
        "n_training_used": n_used,
        "n_training_censored": n_censored,
        "censoring_percent":
            100 * n_censored / n_total
        if n_total else np.nan,
    }

    if n_used < 3:

        result.update({
            "A": np.nan,
            "c": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2_tau": np.nan,
            "R2_log": np.nan,
        })

        threshold_fit_rows.append(result)

        continue

    try:

        fit_result = fit_product_model(
            train_threshold_df
        )

        result.update({
            "A": fit_result["A"],
            "c": fit_result["c"],
            "MAE": fit_result["MAE"],
            "RMSE": fit_result["RMSE"],
            "R2_tau": fit_result["R2_tau"],
            "R2_log":
                float(
                    fit_result["model"].rsquared
                ),
        })

        threshold_fit_objects[
            threshold
        ] = fit_result

    except Exception as exc:

        print(
            f"WARNING: threshold={threshold:.1e} "
            f"fit failed: {exc}"
        )

        result.update({
            "A": np.nan,
            "c": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2_tau": np.nan,
            "R2_log": np.nan,
        })

    threshold_fit_rows.append(
        result
    )

threshold_fit_df = pd.DataFrame(
    threshold_fit_rows
)

print("=" * 80)
print("THRESHOLD-SPECIFIC PRODUCT FITS")
print("=" * 80)

display(threshold_fit_df)


### Cell 8 — Compare exponent changes relative to the baseline threshold


In [ ]:
# ============================================================
# CELL 8 — EXPONENT SENSITIVITY RELATIVE TO 1e-2
# ============================================================

baseline_rows = threshold_fit_df[
    np.isclose(
        threshold_fit_df["threshold"],
        BASELINE_THRESHOLD
    )
]

if len(baseline_rows) != 1:
    raise RuntimeError(
        "CRITICAL: Could not identify exactly one baseline-threshold fit."
    )

baseline_row = baseline_rows.iloc[0]

baseline_A = float(
    baseline_row["A"]
)

baseline_c = float(
    baseline_row["c"]
)

comparison = threshold_fit_df.copy()

comparison["delta_A"] = (
    comparison["A"]
    - baseline_A
)

comparison["relative_A_change_percent"] = (
    100
    * comparison["delta_A"]
    / baseline_A
)

comparison["delta_c"] = (
    comparison["c"]
    - baseline_c
)

comparison["relative_c_change_percent"] = (
    100
    * comparison["delta_c"]
    / abs(baseline_c)
)

print("=" * 80)
print("THRESHOLD SENSITIVITY RELATIVE TO BASELINE 1e-2")
print("=" * 80)

display(
    comparison[
        [
            "threshold",
            "A",
            "c",
            "n_training_used",
            "censoring_percent",
            "R2_tau",
            "MAE",
            "RMSE",
            "delta_A",
            "relative_A_change_percent",
            "delta_c",
            "relative_c_change_percent",
        ]
    ]
)


### Cell 9 — Quantify pointwise changes in \(\tau_{BP}\)


In [ ]:
# ============================================================
# CELL 9 — CONFIGURATION-LEVEL TAU SENSITIVITY
# ============================================================

baseline_tau = threshold_tau_df[
    np.isclose(
        threshold_tau_df["threshold"],
        BASELINE_THRESHOLD
    )
][
    ["n", "k", "tau_BP", "censored"]
].rename(
    columns={
        "tau_BP": "tau_baseline",
        "censored": "baseline_censored",
    }
)

tau_comparison = threshold_tau_df.merge(
    baseline_tau,
    on=["n", "k"],
    how="left"
)

tau_comparison["delta_tau"] = (
    tau_comparison["tau_BP"]
    - tau_comparison["tau_baseline"]
)

tau_comparison["abs_delta_tau"] = (
    tau_comparison["delta_tau"].abs()
)

print("=" * 80)
print("CONFIGURATION-LEVEL THRESHOLD CHANGES")
print("=" * 80)

summary_tau_sensitivity = (
    tau_comparison
    .groupby("threshold")
    .agg(
        mean_abs_delta_tau=(
            "abs_delta_tau",
            "mean"
        ),
        median_abs_delta_tau=(
            "abs_delta_tau",
            "median"
        ),
        max_abs_delta_tau=(
            "abs_delta_tau",
            "max"
        ),
        mean_signed_delta_tau=(
            "delta_tau",
            "mean"
        ),
        n_configurations=(
            "tau_BP",
            "size"
        ),
        n_censored=(
            "censored",
            "sum"
        ),
    )
    .reset_index()
)

print()
display(
    summary_tau_sensitivity
)


### Cell 10 — Check monotonicity of the threshold effect


In [ ]:
# ============================================================
# CELL 10 — MONOTONICITY AUDIT
# ============================================================
#
# Lowering the variance threshold should generally require
# a later threshold crossing, or leave it unchanged.
#
# We check this empirically rather than assuming it.
# ============================================================

print("=" * 80)
print("THRESHOLD MONOTONICITY AUDIT")
print("=" * 80)

thresholds_sorted = sorted(
    THRESHOLDS,
    reverse=True
)

monotonicity_records = []

for (n, k), group in threshold_tau_df.groupby(
    ["n", "k"]
):

    ordered = (
        group
        .set_index("threshold")
        .loc[
            [t for t in thresholds_sorted
             if t in group["threshold"].values]
        ]
        ["tau_BP"]
        .to_numpy()
    )

    # As threshold decreases, tau should not decrease.
    monotonic = np.all(
        np.diff(ordered) >= 0
    )

    monotonicity_records.append({
        "n": int(n),
        "k": int(k),
        "monotonic_threshold_response": bool(
            monotonic
        ),
    })

monotonicity_df = pd.DataFrame(
    monotonicity_records
)

display(monotonicity_df)

violations = monotonicity_df[
    ~monotonicity_df[
        "monotonic_threshold_response"
    ]
]

if len(violations) == 0:
    print(
        "\nPASS: All configurations respond "
        "monotonically to the tested thresholds."
    )
else:
    print(
        f"\nWARNING: {len(violations)} configurations "
        "show non-monotonic threshold response."
    )


### Cell 11 — Plot \(c\) versus threshold


In [ ]:
# ============================================================
# CELL 11 — EXPONENT VS THRESHOLD
# ============================================================

plot_df = threshold_fit_df.dropna(
    subset=["c"]
).copy()

plot_df = plot_df.sort_values(
    "threshold",
    ascending=False
)

plt.figure(figsize=(9, 6))

plt.semilogx(
    plot_df["threshold"],
    plot_df["c"],
    marker="o",
    linewidth=2
)

plt.axvline(
    BASELINE_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label="Baseline threshold"
)

plt.xlabel(
    "Absolute Gradient-Variance Threshold"
)

plt.ylabel(
    "Fitted exponent c"
)

plt.title(
    "Threshold Sensitivity of Scaling Exponent"
)

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "exponent_vs_threshold.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()


### Cell 12 — Plot \(A\) versus threshold


In [ ]:
# ============================================================
# CELL 12 — A VS THRESHOLD
# ============================================================

plot_df = threshold_fit_df.dropna(
    subset=["A"]
).copy()

plot_df = plot_df.sort_values(
    "threshold",
    ascending=False
)

plt.figure(figsize=(9, 6))

plt.semilogx(
    plot_df["threshold"],
    plot_df["A"],
    marker="o",
    linewidth=2
)

plt.axvline(
    BASELINE_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label="Baseline threshold"
)

plt.xlabel(
    "Absolute Gradient-Variance Threshold"
)

plt.ylabel(
    "Fitted prefactor A"
)

plt.title(
    "Threshold Sensitivity of Scaling Prefactor"
)

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "A_vs_threshold.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()


### Cell 13 — Plot censoring versus threshold


In [ ]:
# ============================================================
# CELL 13 — CENSORING VS THRESHOLD
# ============================================================

plt.figure(figsize=(9, 6))

plt.semilogx(
    threshold_fit_df["threshold"],
    threshold_fit_df["censoring_percent"],
    marker="o",
    linewidth=2
)

plt.axvline(
    BASELINE_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label="Baseline threshold"
)

plt.xlabel(
    "Absolute Gradient-Variance Threshold"
)

plt.ylabel(
    "Training Configurations Censored (%)"
)

plt.title(
    "Censoring Introduced by Threshold Choice"
)

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "censoring_vs_threshold.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()


### Cell 14 — Plot representative \(\tau_{BP}\) responses


In [ ]:
# ============================================================
# CELL 14 — REPRESENTATIVE CONFIGURATION SENSITIVITY
# ============================================================

# Automatically choose:
#   - one late-onset configuration
#   - one middle configuration
#   - one early-onset configuration
#
# based entirely on baseline tau.

baseline_full = threshold_tau_df[
    np.isclose(
        threshold_tau_df["threshold"],
        BASELINE_THRESHOLD
    )
].copy()

baseline_full = baseline_full[
    baseline_full["n"].isin(TRAIN_N)
].copy()

baseline_full = baseline_full.sort_values(
    "tau_BP"
)

if len(baseline_full) < 3:
    raise RuntimeError(
        "Not enough baseline configurations for representative plotting."
    )

representatives = [
    baseline_full.iloc[0],
    baseline_full.iloc[len(baseline_full) // 2],
    baseline_full.iloc[-1],
]

for row in representatives:

    n_val = int(row["n"])
    k_val = int(row["k"])

    sub = threshold_tau_df[
        (threshold_tau_df["n"] == n_val)
        &
        (threshold_tau_df["k"] == k_val)
    ].sort_values(
        "threshold",
        ascending=False
    )

    plt.figure(figsize=(9, 6))

    plt.semilogx(
        sub["threshold"],
        sub["tau_BP"],
        marker="o",
        linewidth=2
    )

    plt.axvline(
        BASELINE_THRESHOLD,
        linestyle="--",
        linewidth=1.5,
        label="Baseline threshold"
    )

    plt.xlabel(
        "Absolute Gradient-Variance Threshold"
    )

    plt.ylabel(
        r"$\tau_{BP}$"
    )

    plt.title(
        f"Threshold Sensitivity: n={n_val}, k={k_val}"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"tau_threshold_n{n_val}_k{k_val}.png"
        ),
        dpi=250,
        bbox_inches="tight"
    )

    plt.show()


### Cell 15 — Final numerical summary


In [ ]:
# ============================================================
# CELL 15 — FINAL THRESHOLD SENSITIVITY SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("CHECKPOINT 4 — ABSOLUTE THRESHOLD SENSITIVITY SUMMARY")
print("=" * 90)

print(
    f"\nBaseline threshold: {BASELINE_THRESHOLD:.1e}"
)

print(
    f"Baseline A: {baseline_A:.8f}"
)

print(
    f"Baseline c: {baseline_c:.8f}"
)

print("\nThreshold-specific results:")
display(
    comparison[
        [
            "threshold",
            "A",
            "c",
            "n_training_used",
            "n_training_censored",
            "censoring_percent",
            "R2_tau",
            "MAE",
            "RMSE",
            "relative_c_change_percent",
        ]
    ].sort_values(
        "threshold",
        ascending=False
    )
)

print("\nTAU sensitivity:")
display(
    summary_tau_sensitivity.sort_values(
        "threshold",
        ascending=False
    )
)

print("\nMonotonicity:")
print(
    f"Configurations tested: "
    f"{len(monotonicity_df)}"
)

print(
    f"Monotonicity violations: "
    f"{len(violations)}"
)

print(
    "\nIMPORTANT:"
)

print(
    "This checkpoint does not declare a threshold "
    "'correct' or 'incorrect'."
)

print(
    "It quantifies how the empirical scaling changes "
    "when the absolute operational threshold changes."
)


### Cell 16 — Save all deliverables


In [ ]:
# ============================================================
# CELL 16 — SAVE CHECKPOINT 4 OUTPUTS
# ============================================================

threshold_tau_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_tau_all_configs.csv"
    ),
    index=False
)

tau_pivot.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_tau_pivot.csv"
    ),
    index=False
)

threshold_fit_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_scaling_fits.csv"
    ),
    index=False
)

comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_vs_baseline_comparison.csv"
    ),
    index=False
)

tau_comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "configuration_tau_sensitivity.csv"
    ),
    index=False
)

summary_tau_sensitivity.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "tau_sensitivity_summary.csv"
    ),
    index=False
)

monotonicity_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_monotonicity.csv"
    ),
    index=False
)

checkpoint4_summary = {
    "checkpoint": 4,
    "input_file": INPUT_FILE,
    "baseline_threshold": BASELINE_THRESHOLD,
    "tested_thresholds": THRESHOLDS,
    "training_n": TRAIN_N,
    "held_out_n": HELD_OUT_N,
    "baseline_A": baseline_A,
    "baseline_c": baseline_c,
    "n_thresholds": len(THRESHOLDS),
    "monotonicity_violations": int(
        len(violations)
    ),
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "checkpoint4_summary.json"
    ),
    "w"
) as f:
    json.dump(
        checkpoint4_summary,
        f,
        indent=2
    )

print("=" * 80)
print("CHECKPOINT 4 FILES SAVED")
print("=" * 80)

for fname in sorted(
    os.listdir(OUTPUT_DIR)
):
    print(
        os.path.join(
            OUTPUT_DIR,
            fname
        )
    )

print("\nSTATUS: COMPLETE")


What matters when you send me the results

The most important output is Cell 15, especially:

threshold | A | c | training_used | censoring | R² | MAE | RMSE

We are not going to arbitrarily say “robust” just because \(c\) changes by less than some chosen percentage. We will look at the actual dependence, the amount of censoring, and whether the scaling remains structurally reasonable.

Also, this notebook deliberately keeps \(n=14,16\) out of the fits. The purpose here is sensitivity of the training-domain law, not another validation experiment.

After this, the next major checkpoint is Notebook 5 — \(n\)-Scaled Threshold Diagnostic, which is more important scientifically because it tests whether the apparent \(n\)-dependence could be tied to the fixed absolute threshold.